# Export Tables to Parquet

This notebook exports all tables from the schema as .parquet files to a Unity Catalog volume called 'export'.

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "dev"
VOLUME_NAME = "export"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"

In [ ]:
# Create the export volume if it doesn't exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
print(f"Volume ready: {VOLUME_PATH}")

In [ ]:
# List all tables in the schema
tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
tables = [row.tableName for row in tables_df.collect() if not row.tableName.endswith('_index')]
print(f"Found {len(tables)} tables in {CATALOG}.{SCHEMA}:")
for t in tables:
    print(f"  - {t}")

In [ ]:
# Export each table to a single parquet file using pandas
exported = []
failed = []

for table_name in tables:
    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    output_path = f"{VOLUME_PATH}/{table_name}.parquet"

    try:
        df = spark.table(full_table_name)
        pdf = df.toPandas()
        row_count = len(pdf)

        pdf.to_parquet(output_path, index=False)

        exported.append((table_name, row_count, output_path))
        print(f"Exported {table_name} ({row_count} rows) -> {output_path}")
    except Exception as e:
        failed.append((table_name, str(e)))
        print(f"Failed to export {table_name}: {e}")

print(f"\nExported {len(exported)} tables, {len(failed)} failed")

In [ ]:
# List exported files in the volume
import os

print(f"Files in {VOLUME_PATH}:")
for item in os.listdir(VOLUME_PATH):
    item_path = os.path.join(VOLUME_PATH, item)
    size = os.path.getsize(item_path)
    print(f"  {item}: {size / (1024*1024):.2f} MB")

In [ ]:
# Summary
print("=" * 50)
print("EXPORT SUMMARY")
print("=" * 50)
print(f"\nSuccessfully exported {len(exported)} tables:")
for table_name, row_count, path in exported:
    print(f"  {table_name}: {row_count} rows")

if failed:
    print(f"\nFailed to export {len(failed)} tables:")
    for table_name, error in failed:
        print(f"  {table_name}: {error}")